# Capstone — Which pages should a reviewer look at first?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/susheel123-sketch/Flyrank-Internship-ml/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This notebook mirrors the deployed paper (`docs/index.html`). It re-runs the headline numbers from scratch and **asserts they match** the receipts
committed by Weeks 4–7 (`work/outputs/w0*_metrics.json`), then builds the figures the paper embeds.
**Author:** Azad Wali · **Lane:** 2 — Refresh / Content Opportunity Scoring · **Seed:** 42

In [1]:
import os, sys, json, shutil, subprocess, warnings
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
IN_COLAB = "google.colab" in sys.modules
REPO_URL, REPO_DIR = "https://github.com/susheel123-sketch/Flyrank-Internship-ml", "Flyrank-Internship-ml"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
SEED = 42
R = {w: json.load(open(f"work/outputs/{w}")) for w in
     ["w04_baseline_metrics.json", "w05_model_metrics.json", "w06_audit_metrics.json", "w07_playbook_metrics.json"]}
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
y = (df["trend_direction"] == "down").astype(int).values
groups = df["client_id"].values
print(f"pages {len(df):,} | clients {df['client_id'].nunique()} | declining {y.sum():,} | base rate {y.mean():.3f}")

pages 30,000 | clients 32 | declining 16,262 | base rate 0.542


## 1. Question

**Out of thousands of published pages, which ones should a content reviewer look at first?**

FlyRank publishes and maintains content for many brands. Pages that rank can quietly lose visibility, and a review team
can only open a few dozen a week. FlyRank's product already flags pages with hand-written rules (refresh, CTR-fix, quick-win).
This capstone asks whether a transparent rule and a simple learned ranker can order the review list better than chance —
and how much of any gain survives honest validation.

**Decision supported:** review order. **Unit:** one page. **Output:** a ranked queue with an action, a reason code and review flags.
**Cost of a wrong call:** a reviewer's time on a page that was fine (false alarm), or a slipping page left unnoticed (miss).

In [2]:
print("Label (evaluation only): impressions fell >20% between the last 30 days and the 30 days before.")
print(f"Positives: {y.sum():,} of {len(y):,} ({y.mean():.1%}) -> random ordering scores {y.mean():.2f} at any K.")

Label (evaluation only): impressions fell >20% between the last 30 days and the 30 days before.
Positives: 16,262 of 30,000 (54.2%) -> random ordering scores 0.54 at any K.


## 2. Data

- **Source:** the FlyRank ML Internship starter release — a pseudonymised slice of Google Search Console + GA4 data.
  30,000 pages, 32 clients, one 90-day snapshot plus two 30-day comparison windows.
- **Population:** every page is ≥ 90 days old and had ≥ 1 impression in the window (a built-in filter, disclosed in Limitations).
- **Excluded from all inputs:** `trend_direction`, `trend_pct` (label source) and the last-30 / prev-30 columns (they build the label).
  IDs used only for grouping. No product flags exist in this release.
- **Warehouse:** Dec 2025–Mar 2026 partitions of the ~79M-row daily table, rows with Search Console data only.
  - **Forward test** (`w08_warehouse_forward_test.ipynb`): trained on Dec→(Feb vs Jan), tested on Jan→(Mar vs Feb) for 113,588 pages from 33 unseen clients. The final ranker scored P@100 0.68 and P@500 0.71 vs a 0.33 base rate.
  - **Earlier check** (`w03_*`): scored 0.793 vs a 0.812 base rate.
  - The rest of this notebook re-runs the starter-slice numbers.

In [3]:
# The strict-timeline window: days 61-90 back from export, strictly before the label's 60 days.
w61 = pd.DataFrame({"impr": df["impressions_90d"] - df["impressions_last_30d"] - df["impressions_prev_30d"],
                    "clicks": df["clicks_90d"] - df["clicks_last_30d"] - df["clicks_prev_30d"],
                    "sess": df["sessions_90d"] - df["sessions_last_30d"] - df["sessions_prev_30d"]})
assert (w61 >= 0).all().all()
ctr61 = np.where(w61["impr"] > 0, w61["clicks"] / w61["impr"].replace(0, 1) * 100, 0)
print("Days 61-90 rebuilt exactly (no negatives). Median impressions in that window:", w61["impr"].median())

Days 61-90 rebuilt exactly (no negatives). Median impressions in that window: 308.0


## 3. Methodology

1. **Signal checks (Week 4):** staleness → **MIXED** (pooled gap vanishes inside clients; updates come in client batches).
   CTR below its position band → **CONFIRMED** (positions 1–20, inside 15 of 16 clients).
2. **Baseline (Week 4):** one transparent rule — flag pages on positions 1–20 with ≥500 impressions whose CTR is under half
   their band's median; rank by CTR gap × log(impressions). One reason code, one action.
3. **Models (Week 5):** Logistic Regression, depth-3 tree, Random Forest, Gradient Boosting; probabilities used as a ranking.
4. **Validation (Weeks 5–6):** GroupKFold by client (5 folds) — every page scored by a model that never saw its client.
   Then a **time-aware feature window**: only days 61–90 traffic + page metadata, so no input overlaps the label window.
5. **Leakage checks (Week 6):** planted-leak test (AUC → 1.00, harness works), drop-one tests, timeline table, flag + population scan.
6. **Playbook (Week 7):** rule-based archetypes → actions, per-client cap, human-review gates, monitoring triggers.

In [4]:
CAT = ["competition_level", "content_type", "main_intent"]
def strict_features(d):
    X = pd.DataFrame(index=d.index)
    for c in ["search_volume", "cpc"]:
        X[c] = np.log1p(d[c]).fillna(0); X["has_" + c] = d[c].notna().astype(int)
    X["competition"] = d["competition"].fillna(0)
    X["word_count"] = d["word_count"].fillna(0); X["has_word_count"] = d["word_count"].notna().astype(int)
    X["age_at_cutoff"] = d["content_age_days"] - 60
    X["impr_61_90"] = np.log1p(w61["impr"]); X["clicks_61_90"] = np.log1p(w61["clicks"]); X["sess_61_90"] = np.log1p(w61["sess"])
    X["ctr_61_90"] = ctr61; X["visible_61_90"] = (w61["impr"] > 0).astype(int)
    return X.join(pd.get_dummies(d[CAT].fillna("unknown"), dtype=int))
X = strict_features(df)
FOLDS = list(GroupKFold(5).split(X, y, groups))
for tr, te in FOLDS:
    assert not set(groups[tr]) & set(groups[te])
print(len(X.columns), "strict-timeline features; 5 client-grouped folds, no client overlap (asserted)")

25 strict-timeline features; 5 client-grouped folds, no client overlap (asserted)


## 4. Results (vs baseline)

Re-run here from scratch, then checked against the committed receipts.

In [5]:
def p_at_k(s, k): return float(y[np.argsort(-np.asarray(s), kind="stable")[:k]].mean())
score = np.zeros(len(df)); rule = np.zeros(len(df))
band = pd.cut(df["avg_position"].where(df["avg_position"] > 0), [0, 3, 10, 20], labels=["1-3", "4-10", "11-20"])
imp3, c61 = w61["impr"] * 3, pd.Series(ctr61, index=df.index)
for tr, te in FOLDS:
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(X.iloc[tr], y[tr])
    score[te] = m.predict_proba(X.iloc[te])[:, 1]
    ti, ei = df.index[tr], df.index[te]
    ok = (imp3[ti] >= 500) & band[ti].notna()
    med = c61[ti][ok].groupby(band[ti][ok], observed=True).median()
    e = band[ei].map(med).astype(float)
    flag = (imp3[ei] >= 500) & band[ei].notna() & (c61[ei] < 0.5 * e)
    rule[te] = np.where(flag, (1 - c61[ei] / e).clip(0, 1) * np.log1p(imp3[ei]), 0) + np.log1p(imp3[ei].values) * 1e-6

w6 = {r["setting"]: r for r in R["w06_audit_metrics.json"]["claim_ledger"]}
results = pd.DataFrame([
    ("Random order", "—", y.mean(), np.nan),
    ("Week-4 rule (as built, all 30k)", "none", R["w04_baseline_metrics.json"]["precision_at_k"]["50"], np.nan),
    ("Week-4 rule (strict timeline)", "client-grouped", p_at_k(rule, 50), np.nan),
    ("Gradient Boosting, Week-5 features", "RANDOM split", w6["Week-5 GB, random split"]["pooled P@50"], w6["Week-5 GB, random split"]["AUC"]),
    ("Gradient Boosting, Week-5 features", "client-grouped", w6["Week-5 GB, grouped split"]["pooled P@50"], w6["Week-5 GB, grouped split"]["AUC"]),
    ("Gradient Boosting, strict timeline", "client-grouped", w6["GB, grouped + strict timeline"]["pooled P@50"], w6["GB, grouped + strict timeline"]["AUC"]),
    ("Logistic Regression, strict timeline (FINAL)", "client-grouped", p_at_k(score, 50), roc_auc_score(y, score)),
], columns=["ranker", "split", "P@50", "AUC"]).round(3)
display(results)
assert abs(p_at_k(score, 50) - R["w07_playbook_metrics.json"]["reference_for_monitoring"]["precision_at_50"]) < 1e-9
assert abs(p_at_k(rule, 50) - R["w07_playbook_metrics.json"]["reference_for_monitoring"]["rule_bar_precision_at_50"]) < 1e-9
print("Re-run matches the Week-7 receipts (asserted).")
pk = pd.DataFrame({"LR strict (final)": [p_at_k(score, k) for k in [10, 20, 50, 100, 200, 500]],
                   "rule strict": [p_at_k(rule, k) for k in [10, 20, 50, 100, 200, 500]]}, index=[10, 20, 50, 100, 200, 500]).round(3)
pk["base rate"] = round(y.mean(), 3); display(pk)

,ranker,split,P@50,AUC
0,Random order,—,0.542,NaN
1,"Week-4 rule (as built, all 30k)",none,0.640,NaN
2,Week-4 rule (strict timeline),client-grouped,0.700,NaN
3,"Gradient Boosting, Week-5 features",RANDOM split,0.960,0.781
4,"Gradient Boosting, Week-5 features",client-grouped,0.840,0.685
5,"Gradient Boosting, strict timeline",client-grouped,0.600,0.612
6,"Logistic Regression, strict timeline (FINAL)",client-grouped,0.860,0.661


Re-run matches the Week-7 receipts (asserted).


,LR strict (final),rule strict,base rate
10,0.900,0.700,0.542
20,0.900,0.650,0.542
50,0.860,0.700,0.542
100,0.850,0.750,0.542
200,0.850,0.720,0.542
500,0.792,0.738,0.542


**Reading it:** the honest final ranker puts 86% declining pages in its top 50, against 70% for the rule and 54% at random.
The random-split number (0.96) is what a standard workflow would have reported; the Gradient Boosting win disappears once inputs
are restricted to the time before the label window.

## 5. Limitations

- **One snapshot, same-window label.** "Declining" = impressions fell between two recent 30-day windows. This is decision-support for review order, not a forecast.
- **No causal claim.** Nothing in this data was randomly refreshed or rewritten, so no action here is shown to prevent decline.
- **Precision ≠ value.** The top 50 covered only 10 earlier-window clicks; a click-weighted lane covered 21,167 clicks but declined at 48% (below random).
- **Client effects dominate.** 7 of 32 clients sit below 20% or above 80% declining; raw model order put 84% of the top 200 in one client (fixed by a per-client cap).
- **Label noise.** 15% of declines rest on fewer than 20 earlier impressions.
- **Zero-click concentration.** 82% of the top 50 had zero earlier clicks — possibly tracking gaps; the playbook makes a tracking check mandatory.
- **Population.** Only pages ≥90 days old with ≥1 impression; starter slice only, not the full warehouse.

In [6]:
print("Label-noise check:", int(((df["impressions_prev_30d"] < 20) & (y == 1)).sum()), "of", int(y.sum()), "declines rest on <20 earlier impressions")
cb = pd.Series(y).groupby(groups).mean()
print("Clients with base rate <0.2 or >0.8:", int(((cb < 0.2) | (cb > 0.8)).sum()), "of", len(cb))
print("Value check (Week 7):", {k: R["w07_playbook_metrics.json"]["cost_value"][k]["earlier_window_clicks_covered"] for k in ["capped_top_50", "value_lane_50", "random_50"]})

Label-noise check: 2445 of 16262 declines rest on <20 earlier impressions
Clients with base rate <0.2 or >0.8: 7 of 32
Value check (Week 7): {'capped_top_50': 10, 'value_lane_50': 21167, 'random_50': 189}


## 6. Ranked recommendations

1. **Run two short lists:** a capped risk queue (top 50, max 10 pages per client) and a value lane (top 20 by score × earlier clicks).
2. **Check tracking before editing** any page with zero clicks on ≥300 impressions.
3. **Skip pages already updated in the last 60 days**; wait for that change to show.
4. **Treat a client with many queue pages as one site-level case**, not many page fixes.
5. **Never automate** rewrites, merges, redirects, deletions or client-facing "your content is declining" messages from this score.
6. **Next modelling step:** a click-weighted label so risk and value point at the same pages; validate on a forward window in the warehouse.

In [7]:
am = pd.DataFrame(R["w07_playbook_metrics.json"]["archetype_map"])[["archetype", "action", "n", "decline_rate"]]
am["decline_rate"] = am["decline_rate"].round(2); display(am.sort_values("decline_rate", ascending=False))

,archetype,action,n,decline_rate
5,zero_click_visible,verify_tracking_then_snippet_review,3809,0.68
1,low_ctr_page_1_2,rewrite_title_meta,1209,0.67
4,young_fading_risk,content_refresh_review,6971,0.67
0,large_established,protect_light_review,1150,0.56
3,steady_mid,monitor,6140,0.49
2,quiet_page,no_action_batch_monitor,10721,0.42


## 7. Artifacts the paper embeds

In [8]:
os.makedirs("docs/img", exist_ok=True); os.makedirs("work/figures", exist_ok=True)
lab = ["Random\norder", "Rule\n(strict)", "GB\nrandom split", "GB\ngrouped", "GB grouped\n+ strict", "LR grouped\n+ strict"]
val = [y.mean(), p_at_k(rule, 50), w6["Week-5 GB, random split"]["pooled P@50"], w6["Week-5 GB, grouped split"]["pooled P@50"],
       w6["GB, grouped + strict timeline"]["pooled P@50"], p_at_k(score, 50)]
col = ["#B8B8B8", "#E0A458", "#C9CFD8", "#8DA1BF", "#8DA1BF", "#2E4C7A"]
fig, ax = plt.subplots(figsize=(7, 3.6))
bars = ax.bar(lab, val, color=col)
for b, v in zip(bars, val): ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f"{v:.2f}", ha="center", fontsize=9)
ax.axhline(y.mean(), ls="--", c="grey", lw=1)
ax.set_ylim(0, 1.08); ax.set_ylabel("share of top 50 that declined (P@50)")
ax.set_title("How much of the score survives honest validation", fontsize=10); ax.tick_params(axis="x", labelsize=8)
ax.spines[["top", "right"]].set_visible(False); fig.tight_layout()
fig.savefig("work/figures/capstone_validation_story.png", dpi=150); fig.savefig("docs/img/validation_story.png", dpi=150); plt.show()
for src, dst in [("work/figures/w07_precision_at_k.png", "docs/img/precision_at_k.png"),
                 ("work/figures/w07_decline_by_age.png", "docs/img/decline_by_age.png")]:
    shutil.copy(src, dst)
print(sorted(os.listdir("docs/img")))

['decline_by_age.png', 'precision_at_k.png', 'validation_story.png']


## ML-12 — 5-minute demo outline (Week-8 showcase)

| Time | Beat | What I say / show |
|---|---|---|
| 0:00–0:45 | **Question** | "FlyRank keeps content live for many brands. Pages quietly lose visibility, and a reviewer can open maybe 50 a week. Which 50?" |
| 0:45–1:45 | **Method** | Hand-written rule first (low CTR for its position — the signal behind FlyRank's CTR-fix flag). Then a simple model. Every page scored by a model that never saw its client, using only data from *before* the window the label is measured in. |
| 1:45–3:00 | **One chart** | *How much of the score survives honest validation* (`docs/img/validation_story.png`). Walk left to right: 0.96 with a random split → 0.84 grouped by client → 0.60 when boosting loses label-window features → 0.86 for plain Logistic Regression that never relied on them. |
| 3:00–4:00 | **One honest result** | "The final ranker puts 86% declining pages in its top 50, vs 70% for the rule and 54% at random. But those 50 pages had only 10 clicks between them — it finds decline, not value." |
| 4:00–4:45 | **One recommendation** | Two short lists: a capped risk queue + a click-weighted value lane, each page through a tracking check and a "recently updated?" check first. Nothing auto-edited. |
| 4:45–5:00 | **Close** | "The biggest lesson: the honest split changed which model I'd ship." Link to the paper. |

**Backup answers:** *Does it hold beyond the starter data?* — yes, at the top of the list. On the full warehouse, a month later and on 33 unseen clients, 68–71% of the top 100–1,000 pages declined vs 33% at random; the rule and gradient boosting did not hold up.
*Did refreshing help?* — this data can't say; no page was randomly refreshed.

## ML-12 — Two shareable cuts

**Social post (methodology):**

> My FlyRank internship capstone asked a simple question: out of thousands of live pages, which should a content reviewer open first?
>
> The lesson was about validation, not models. With a random train/test split, my best model put 96% declining pages in its top 50. Grouped by client, 84%. Once I removed every input that overlapped the window the label is measured in, that model fell to 60% — no better than a hand-written rule.
>
> A plain logistic regression held at 86% (vs 54% at random) because it never leaned on those inputs. Same data, very different story depending on how honestly you test.
>
> Then the real test: the full FlyRank warehouse, one month later, on 33 clients the model had never seen. The simple model's top 100–1,000 pages declined at 68–71% vs 33% at random — while the rule and the boosted model fell to or below random.
>
> One more honest catch: the pages it flags most confidently had almost no clicks. Finding decline isn't the same as finding value — that's the next thing to fix.
>
> Paper + notebooks: https://susheel123-sketch.github.io/Flyrank-Internship-ml/ · Data: FlyRank ML Internship release (flyrank.ai)

**Employer-facing summary (3 sentences):**

> I built a ranked review queue that tells a content team which web pages to check first for declining search visibility, with a reason code, a suggested action and human-review flags for every page.
> It was built on 30,000 real, pseudonymised pages from 32 client sites in the FlyRank ML Internship dataset (Google Search Console + GA4), then forward-tested on 113,588 pages from the full warehouse — a later month, and 33 clients the model never saw.
> The final model's top 100–1,000 pages declined at 68–71% vs 33% at random, while a hand-written rule and a stronger-looking boosted model fell to or below random — so I shipped the simpler model and documented why.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (executed before commit)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.